# Embedding multimodal data for similarity search using 🤗 transformers, 🤗 datasets and FAISS

_Authored by: [Merve Noyan](https://huggingface.co/merve)_

Embeddings are semantically meaningful compressions of information. They can be used to do similarity search, zero-shot classification or simply train a new model. Use cases for similarity search include searching for similar products in e-commerce, content search in social media and more.
This notebook walks you through using 🤗transformers, 🤗datasets and FAISS to create and index embeddings from a feature extraction model to later use them for similarity search.
Let's install necessary libraries.

In [ ]:
!pip install dill>=0.3.9 dill>=0.3.9
!pip install torchvision 
!pip install -q datasets faiss-gpu transformers sentencepiece

For this tutorial, we will use [CLIP model](https://huggingface.co/openai/clip-vit-base-patch16) to extract the features. CLIP is a revolutionary model that introduced joint training of a text encoder and an image encoder to connect two modalities.

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer
import faiss
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

model = AutoModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
processor = AutoImageProcessor.from_pretrained("openai/clip-vit-base-patch16")
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch16")

## Loading the embeddings
We can save the dataset with embeddings with `save_faiss_index`.


In [ ]:
import sagemaker

bucket = "programboy-sagemaker-example"
sess = sagemaker.Session(default_bucket=bucket)

sess.download_data(path='./download/dataset', bucket=sess.default_bucket(), key_prefix='dataset')

sess.download_data(path='./download/index', bucket=sess.default_bucket(), key_prefix='index')

In [ ]:
from datasets import load_from_disk

ds = load_from_disk("./download/dataset")
ds.load_faiss_index('embeddings', './download/index/embeddings.faiss')
ds.load_faiss_index('image_embeddings', './download/index/image_embeddings.faiss')

In [ ]:
def downscale_images(image):
  width = 200
  ratio = (width / float(image.size[0]))
  height = int((float(image.size[1]) * float(ratio)))
  img = image.resize((width, height), Image.Resampling.LANCZOS)
  return img

In [ ]:
import requests
# image of a beaver
url = "https://datasets-server.huggingface.co/cached-assets/yusuf802/leaf-images/--/21a4c1f7ca0e538f4bf2be0aa92e385d7cd041c0/--/default/train/33/image/image.jpg?Expires=1742397789&Signature=u-BwDD5wYRgMWaqhHhB~TlaTKtd4-MYgGSKJhYnQ0DxofiKNMx-5oY7UOOHK6wfypKQIGgXp6dTrD2D2uvuJbDYGfLjCHvB3weLHsyiNaL5g1V4odZbNh3r0ly1hDm6fOR0Ao768kHdVDCsSFrm4QDkIwPIZhjwWd7rFjQbOfM0lzPghrQ49RV5~NCLL9HTDbPlQGtri0HIPBuuGZdsy2yVIed5hJyX~eRhJ1JZIdBs6ajlFbQy0OwCx9LEUqX82ndPYSq2W53fEcVZE7Y7guP-Enty8Wv1yJKWgb3cUNR3wgAEB19icHQRBpt6xHX1QaNLmTudZhkVQz1z0ZTe~jg__&Key-Pair-Id=K3EI6M078Z3AC3"
image = Image.open(requests.get(url, stream=True).raw)
display(downscale_images(image))

In [ ]:
img_embedding = model.get_image_features(**processor([image], return_tensors="pt", truncation=True).to("cuda"))[0].detach().cpu().numpy()
scores, retrieved_examples = ds.get_nearest_examples('image_embeddings', img_embedding, k=1)

In [ ]:
images = [downscale_images(image) for image in retrieved_examples["image"]]
# see the closest text and image
print(retrieved_examples["label"])
display(images[0])